In [24]:
import os
import pandas as pd
import xarray as xr
import s3fs
from pygeohydro import NWIS
from pynhd import NLDI


In [39]:
# Read the list of USGS sites from Station_List.csv in the Outputs folder.
# The CSV must have a column named 'site_no' (or update site_col to match your heading).

from pathlib import Path

# Paths
path_data    = "Outputs"
path_figures = "Figures"
path_nwm_out = os.path.join(path_data, "NWM_ins_data")

# Create folders if they do not already exist.
for folder in [path_data, path_figures, path_nwm_out]:
    if not os.path.exists(folder):
        os.makedirs(folder)

# Read station list — update 'site_no' to match the exact column heading in your CSV.
station_file = os.path.join(path_data, "Station_List.csv")

if not os.path.exists(station_file):
    raise FileNotFoundError(
        f"Could not find {station_file}\n"
        f"Current working directory: {os.getcwd()}\n"
        f"Files here: {os.listdir('.')}"
    )

stations_df = pd.read_csv(station_file, dtype=str)
site_col    = stations_df.columns[0]   # auto-detects first column as site ID
# Pad USGS site IDs back to 8 digits with leading zeros
stations_df[site_col] = stations_df[site_col].str.strip().str.zfill(8)
print(f"Using column '{site_col}' as USGS site IDs")

USGS_sites = stations_df[site_col].str.strip().tolist()
print(f"Total stations to process: {len(USGS_sites)}")
print(USGS_sites)

Using column 'Station_List' as USGS site IDs
Total stations to process: 101
['07148400', '07152000', '07153000', '07157950', '07158000', '07159100', '07159750', '07160000', '07160350', '07160500', '07161450', '07164600', '07165562', '07165565', '07165570', '07171000', '07174400', '07175500', '07176000', '07176500', '07177650', '07177800', '07185000', '07188000', '07189540', '07189542', '07191000', '07191220', '07191500', '07195855', '07195865', '07196000', '07196500', '07197000', '07197360', '07228500', '07229050', '07229200', '07229300', '07230000', '07230500', '07231000', '07231500', '07234000', '07237500', '07238000', '07239300', '07239450', '07239500', '07239700', '07240000', '07241000', '07241520', '07241550', '07242000', '07242380', '07243500', '07245000', '07247015', '07247250', '07247500', '07249413', '07249455', '07249800', '07249985', '07300500', '07301110', '07301420', '07303000', '07305000', '07307028', '07311000', '07311500', '07315700', '07316500', '07324200', '07324400',

In [40]:
nwis = NWIS()
info = nwis.get_info({"sites": "07148400"})
print(info[["site_no", "station_nm", "dec_lat_va", "dec_long_va"]])

    site_no                            station_nm  dec_lat_va  dec_long_va
0  07148400  Salt Fork Arkansas River Nr Alva, OK   36.815031   -98.648139


In [41]:
# Retrieve station coordinates from the USGS inventory for each site,
# then map each to the NHDPlus COMID using NLDI.

nwis = NWIS()
site_info = {}   # stores {USGSsite: {"lon": ..., "lat": ..., "comid": ...}}

for USGSsite in USGS_sites:
    try:
        query = {
            "sites": USGSsite,
            "hasDataTypeCd": "iv",
            "outputDataTypeCd": "iv",
            "parameterCd": "00060",
        }
        info = nwis.get_info(query)

        # Extract as plain Python floats (fixes "tuple or list of tuples" error)
        lon = float(info["dec_long_va"].iloc[0])
        lat = float(info["dec_lat_va"].iloc[0])

        # Pass coords as a tuple of floats
        nldi = NLDI()
        comid_result = nldi.comid_byloc((lon, lat))
        comid_site   = int(comid_result["comid"].iloc[0])

        site_info[USGSsite] = {"lon": lon, "lat": lat, "comid": comid_site}
        print(f"Site {USGSsite} → COMID {comid_site}")

    except Exception as e:
        print(f"  WARNING: Could not process site {USGSsite}: {e}")
        site_info[USGSsite] = {"lon": None, "lat": None, "comid": None}

Site 07148400 → COMID 21028210
Site 07152000 → COMID 20988263
Site 07153000 → COMID 20972352
Site 07157950 → COMID 21046034
Site 07158000 → COMID 21047070
Site 07159100 → COMID 255558
Site 07159750 → COMID 254068
Site 07160000 → COMID 255270
Site 07160350 → COMID 250456
Site 07160500 → COMID 251506
Site 07161450 → COMID 496568
Site 07164600 → COMID 367965
Site 07165562 → COMID 368019
Site 07165565 → COMID 367999
Site 07165570 → COMID 369995
Site 07171000 → COMID 21798265
Site 07174400 → COMID 21784444
Site 07175500 → COMID 21784710
Site 07176000 → COMID 833464
Site 07176500 → COMID 847886
Site 07177650 → COMID 847326
Site 07177800 → COMID 847340
Site 07185000 → COMID 24778185
Site 07188000 → COMID 7592691
Site 07189540 → COMID 7583211
Site 07189542 → COMID 7583213
Site 07191000 → COMID 21770039
Site 07191220 → COMID 21771103
Site 07191500 → COMID 21773127
Site 07195855 → COMID 397590
Site 07195865 → COMID 397706
Site 07196000 → COMID 399514
Site 07196500 → COMID 400380
Site 07197000 → 

In [42]:
# save the site info to a CSV for reference in Outputs folder
site_info_df = pd.DataFrame.from_dict(site_info, orient="index")
site_info_df.to_csv("Outputs/site_info.csv")

In [44]:
# Open the NWM Retrospective v3.0 instantaneous (hourly) channel routing output
# from the NOAA public S3 bucket.

url = "s3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/chrtout.zarr"

fs    = s3fs.S3FileSystem(anon=True)
store = xr.open_zarr(
    s3fs.S3Map(url, s3=fs),
    consolidated=False
)

print(store)
print("\nStreamflow variable:")
print(store["streamflow"])

<xarray.Dataset> Size: 51TB
Dimensions:         (feature_id: 2776734, time: 385704)
Coordinates:
    elevation       (feature_id) float32 11MB dask.array<chunksize=(2776734,), meta=np.ndarray>
  * feature_id      (feature_id) int64 22MB 101 179 ... 1180001803 1180001804
    gage_id         (feature_id) |S15 42MB dask.array<chunksize=(2776734,), meta=np.ndarray>
    latitude        (feature_id) float32 11MB dask.array<chunksize=(2776734,), meta=np.ndarray>
    longitude       (feature_id) float32 11MB dask.array<chunksize=(2776734,), meta=np.ndarray>
    order           (feature_id) int32 11MB dask.array<chunksize=(2776734,), meta=np.ndarray>
  * time            (time) datetime64[ns] 3MB 1979-02-01T01:00:00 ... 2023-02-01
Data variables:
    crs             |S1 1B ...
    qBtmVertRunoff  (time, feature_id) float64 9TB dask.array<chunksize=(672, 30000), meta=np.ndarray>
    qBucket         (time, feature_id) float64 9TB dask.array<chunksize=(672, 30000), meta=np.ndarray>
    qSfcLatRunof

In [46]:
# For each USGS site, extract instantaneous NWM streamflow using the COMID,
# for the period January 2000 to January 2023.
# Save individual CSV files to Outputs/NWM_ins_data/.
# Already downloaded files are skipped on re-run.
# Failed sites are logged to a failed_sites.csv for review.

# --- Define time period ---
start_date = "2000-01-01"
end_date   = "2023-01-31"

failed_log_path   = os.path.join(path_nwm_out, "failed_sites.csv")
summary_rows      = []

# Load existing failed log if present
if os.path.exists(failed_log_path):
    failed_df_existing = pd.read_csv(failed_log_path, dtype=str)
    previously_failed  = set(failed_df_existing["USGS_site"].tolist())
else:
    previously_failed = set()

for USGSsite, info in site_info.items():

    out_file = os.path.join(path_nwm_out, f"{USGSsite}_NWM_inst.csv")

    # --- Skip if already successfully downloaded ---
    if os.path.exists(out_file):
        print(f"  SKIP (already exists): {out_file}")
        summary_rows.append({"USGS_site": USGSsite, "COMID": info["comid"],
                              "status": "skipped - already downloaded",
                              "n_records": len(pd.read_csv(out_file))})
        continue

    # --- Skip if no COMID was found in Cell 4 ---
    if info["comid"] is None:
        print(f"  SKIP (no COMID): {USGSsite}")
        summary_rows.append({"USGS_site": USGSsite, "COMID": None,
                              "status": "skipped - no COMID", "n_records": 0})
        continue

    comid_site = info["comid"]

    try:
        print(f"Downloading NWM data for site {USGSsite} (COMID {comid_site})...")
        print(f"  Time period: {start_date} to {end_date}")

        # Select the reach by COMID and filter by time period.
        ds_site = store.sel(
            feature_id = comid_site,
            time       = slice(start_date, end_date)
        )
        df_site = ds_site[["streamflow"]].to_dataframe().reset_index()

        # Add identifiers.
        df_site.insert(0, "USGS_site", USGSsite)
        df_site.insert(1, "COMID",     comid_site)

        # Keep only the relevant columns.
        df_site = df_site[["USGS_site", "COMID", "time", "streamflow"]]

        # Save one CSV per station.
        df_site.to_csv(out_file, index=False)

        summary_rows.append({"USGS_site": USGSsite, "COMID": comid_site,
                              "status": "success", "n_records": len(df_site)})

        print(f"  Saved {len(df_site):,} records → {out_file}")

        # Remove from failed log if it was previously failing
        previously_failed.discard(USGSsite)

    except Exception as e:
        print(f"  ERROR for {USGSsite}: {e}")
        summary_rows.append({"USGS_site": USGSsite, "COMID": comid_site,
                              "status": f"failed: {e}", "n_records": 0})

        # Add to failed log
        previously_failed.add(USGSsite)

        # Remove any partial/corrupt file that may have been written
        if os.path.exists(out_file):
            os.remove(out_file)
            print(f"  Removed partial file: {out_file}")

# --- Save download summary ---
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(path_nwm_out, "download_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\nDownload summary saved → {summary_path}")
print(summary_df)

# --- Update failed sites log ---
if previously_failed:
    failed_df = pd.DataFrame({"USGS_site": sorted(previously_failed)})
    failed_df.to_csv(failed_log_path, index=False)
    print(f"\nFailed sites log updated → {failed_log_path}")
    print(f"Failed sites: {sorted(previously_failed)}")
else:
    if os.path.exists(failed_log_path):
        os.remove(failed_log_path)
    print("\nAll sites processed successfully. No failed sites.")

  Time period: 2000-01-01 to 2023-01-31
  Saved 202,368 records → Outputs\NWM_ins_data\07148400_NWM_inst.csv
  Time period: 2000-01-01 to 2023-01-31
  Saved 202,368 records → Outputs\NWM_ins_data\07152000_NWM_inst.csv
  Time period: 2000-01-01 to 2023-01-31
  Saved 202,368 records → Outputs\NWM_ins_data\07153000_NWM_inst.csv
  Time period: 2000-01-01 to 2023-01-31
  Saved 202,368 records → Outputs\NWM_ins_data\07157950_NWM_inst.csv
  Time period: 2000-01-01 to 2023-01-31
  Saved 202,368 records → Outputs\NWM_ins_data\07158000_NWM_inst.csv
  Time period: 2000-01-01 to 2023-01-31
  Saved 202,368 records → Outputs\NWM_ins_data\07159100_NWM_inst.csv
  Time period: 2000-01-01 to 2023-01-31
  Saved 202,368 records → Outputs\NWM_ins_data\07159750_NWM_inst.csv
  Time period: 2000-01-01 to 2023-01-31
  Saved 202,368 records → Outputs\NWM_ins_data\07160000_NWM_inst.csv
  Time period: 2000-01-01 to 2023-01-31
  Saved 202,368 records → Outputs\NWM_ins_data\07160350_NWM_inst.csv
  Time period: 2000